# LeetCode 1176 — Diet Plan Performance bằng Python

**Link đề:** https://leetcode.com/problems/diet-plan-performance/
**Độ khó:** Easy · **Chủ đề:** Array, Sliding Window

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def dietPlanPerformance(self, calories: List[int], k: int, lower: int, upper: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **cửa sổ liên tiếp độ dài k** (window) trên mảng và có bao nhiêu cửa sổ: `n - k + 1`.
2. Baseline: tính tổng từng cửa sổ bằng `sum(calories[i:i+k])` — `O(n·k)`.
3. Kỹ thuật **sliding window cố định**: tổng mới = tổng cũ **+ phần tử vào − phần tử ra** → `O(n)`.
4. Kỹ thuật **prefix sum** (tổng tiền tố): tổng đoạn `[i, j)` = `pre[j] - pre[i]`.
5. Trace, chứng minh invariant, test đối chiếu, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và ví dụ

Một người ăn kiêng ghi lại `calories[i]` là lượng calo ăn vào ngày `i`.

Với **mỗi** chuỗi `k` ngày liên tiếp `calories[i], ..., calories[i+k-1]` (với mọi `0 <= i <= n-k`), gọi `T` là tổng calo của `k` ngày đó:
- Nếu `T < lower` → **mất 1 điểm** (ăn quá ít).
- Nếu `T > upper` → **được 1 điểm** (ăn quá nhiều).
- Ngược lại (`lower <= T <= upper`) → điểm không đổi.

Ban đầu có 0 điểm. Trả về tổng điểm sau khi xét hết các cửa sổ (có thể âm).

| `calories` | `k` | `lower` | `upper` | Output |
|---|---:|---:|---:|---:|
| `[1,2,3,4,5]` | 1 | 3 | 3 | `0` |
| `[3,2]` | 2 | 0 | 1 | `1` |
| `[6,5,0,0]` | 2 | 1 | 5 | `0` |

**Constraints (theo hiểu biết):**
- `1 <= k <= calories.length <= 10^5`
- `0 <= calories[i] <= 20000`
- `0 <= lower <= upper`

### Giải thích ví dụ

**Ví dụ 1** (`k = 1`): mỗi cửa sổ là 1 ngày. `1 < 3` (−1), `2 < 3` (−1), `3` (0), `4 > 3` (+1), `5 > 3` (+1) → tổng `0`.

**Ví dụ 2:** chỉ một cửa sổ `[3,2]`, `T = 5 > 1` → `+1`.

**Ví dụ 3:** cửa sổ `[6,5]` = 11 > 5 (+1); `[5,0]` = 5 (0); `[0,0]` = 0 < 1 (−1) → `0`.

In [1]:
from typing import List

calories, k = [6, 5, 0, 0], 2
n = len(calories)
print("số cửa sổ =", n - k + 1)
for i in range(n - k + 1):
    print(f"cửa sổ bắt đầu i={i}: {calories[i:i + k]} tổng = {sum(calories[i:i + k])}")

số cửa sổ = 3
cửa sổ bắt đầu i=0: [6, 5] tổng = 11
cửa sổ bắt đầu i=1: [5, 0] tổng = 5
cửa sổ bắt đầu i=2: [0, 0] tổng = 0


## 2. Nền tảng Python cần biết

### 2.1. Slicing list `a[i:i+k]`

Lấy `k` phần tử bắt đầu từ index `i` (tới `i+k-1`). Tạo **list mới** → tốn `O(k)` thời gian và bộ nhớ.

In [2]:
a = [10, 20, 30, 40, 50]
print(a[0:3], a[1:4], a[2:5])
print(a[3:3 + 3])      # vượt cuối thì chỉ lấy tới hết, KHÔNG lỗi

[10, 20, 30] [20, 30, 40] [30, 40, 50]
[40, 50]


### 2.2. Có bao nhiêu cửa sổ độ dài k?

Index bắt đầu `i` chạy từ `0` tới `n - k` → **`n - k + 1`** cửa sổ. Vì vậy vòng lặp là `range(n - k + 1)`.

| n | k | các i | số cửa sổ |
|---|---|---|---|
| 5 | 1 | 0..4 | 5 |
| 5 | 5 | 0 | 1 |
| 4 | 2 | 0..2 | 3 |

In [3]:
for n, k in [(5, 1), (5, 5), (4, 2)]:
    print(n, k, list(range(n - k + 1)))

5 1 [0, 1, 2, 3, 4]
5 5 [0]
4 2 [0, 1, 2]


### 2.3. `if / elif` và điểm số

```python
if T < lower:   points -= 1
elif T > upper: points += 1
```
Vì `lower <= upper`, hai điều kiện không thể cùng đúng — `elif` thể hiện rõ điều đó.

### 2.4. Prefix sum với `itertools.accumulate`

`pre[j]` = tổng `j` phần tử đầu. Thêm `0` ở đầu để `pre[0] = 0`. Khi đó tổng đoạn `a[i:j] = pre[j] - pre[i]`.

In [4]:
from itertools import accumulate

a = [6, 5, 0, 0]
pre = [0] + list(accumulate(a))
print("pre =", pre)
print("tổng a[1:3] =", pre[3] - pre[1], "| kiểm tra:", sum(a[1:3]))

pre = [0, 6, 11, 11, 11]
tổng a[1:3] = 5 | kiểm tra: 5


## 3. Cách 1 — Baseline: cộng lại từng cửa sổ

In [5]:
def diet_brute(calories: List[int], k: int, lower: int, upper: int) -> int:
    points = 0
    for i in range(len(calories) - k + 1):
        T = sum(calories[i:i + k])        # O(k) mỗi lần
        if T < lower:
            points -= 1
        elif T > upper:
            points += 1
    return points

print(diet_brute([1, 2, 3, 4, 5], 1, 3, 3))   # 0
print(diet_brute([3, 2], 2, 0, 1))            # 1
print(diet_brute([6, 5, 0, 0], 2, 1, 5))      # 0

0
1
0


### Nút thắt

Có `n - k + 1` cửa sổ, mỗi cửa sổ cộng `k` số → `O((n - k + 1)·k)`. Xấu nhất khi `k ≈ n/2`: `≈ n²/4` = `2.5·10^9` phép cộng với `n = 10^5` → **quá chậm**.

Hai cửa sổ liền nhau **trùng nhau `k - 1` phần tử** — ta đang cộng lại những thứ đã cộng.

```text
cửa sổ i   : [a_i, a_{i+1}, ..., a_{i+k-1}]
cửa sổ i+1 :      [a_{i+1}, ..., a_{i+k-1}, a_{i+k}]
```

In [6]:
import time
big = [i % 7 for i in range(4000)]
t0 = time.perf_counter()
diet_brute(big, 2000, 5000, 7000)
print(f"brute n=4000, k=2000: {time.perf_counter() - t0:.3f} giây")

brute n=4000, k=2000: 0.026 giây


## 4. Insight — Trượt cửa sổ

Từ cửa sổ `i` sang `i + 1`: **bỏ** `a[i]` (ra khỏi bên trái), **thêm** `a[i + k]` (vào bên phải).

```text
T(i+1) = T(i) - a[i] + a[i+k]
```

Ví dụ `[6,5,0,0]`, `k=2`: `T0 = 11`; `T1 = 11 - 6 + 0 = 5`; `T2 = 5 - 5 + 0 = 0`.

In [7]:
a, k = [6, 5, 0, 0], 2
T = sum(a[:k])
print("T0 =", T)
for i in range(k, len(a)):
    T = T + a[i] - a[i - k]
    print(f"thêm a[{i}]={a[i]}, bỏ a[{i - k}]={a[i - k]} -> T = {T}")

T0 = 11
thêm a[2]=0, bỏ a[0]=6 -> T = 5
thêm a[3]=0, bỏ a[1]=5 -> T = 0


## 5. Cách 2 — Lời giải chính: sliding window

**Mổ xẻ:**
1. Tính `T = sum(calories[:k])` cho cửa sổ đầu, chấm điểm.
2. Với `i` từ `k` tới `n - 1`: `T += calories[i] - calories[i - k]`, chấm điểm.
3. Trả `points`.

Ở đây `i` là **index phần tử vừa vào**; phần tử vừa ra là `i - k`.

In [8]:
def score(T: int, lower: int, upper: int) -> int:
    if T < lower:
        return -1
    if T > upper:
        return 1
    return 0

def diet_sliding(calories: List[int], k: int, lower: int, upper: int) -> int:
    T = sum(calories[:k])
    points = score(T, lower, upper)
    for i in range(k, len(calories)):
        T += calories[i] - calories[i - k]
        points += score(T, lower, upper)
    return points

print(diet_sliding([1, 2, 3, 4, 5], 1, 3, 3))
print(diet_sliding([3, 2], 2, 0, 1))
print(diet_sliding([6, 5, 0, 0], 2, 1, 5))

0
1
0


Tách hàm `score` giúp code chính ngắn và tránh lặp lại khối `if/elif` hai lần (cửa sổ đầu và các cửa sổ sau).

## 6. TRACE từng bước

`calories = [1,2,3,4,5]`, `k = 3`, `lower = 6`, `upper = 10`.

| cửa sổ | vào | ra | T | so sánh | điểm | tổng điểm |
|---|---|---|---:|---|---:|---:|
| [1,2,3] | — | — | 6 | 6 ≤ 6 ≤ 10 | 0 | 0 |
| [2,3,4] | 4 | 1 | 9 | trong khoảng | 0 | 0 |
| [3,4,5] | 5 | 2 | 12 | 12 > 10 | +1 | 1 |

In [9]:
def trace_diet(calories: List[int], k: int, lower: int, upper: int) -> int:
    T = sum(calories[:k])
    s = score(T, lower, upper)
    points = s
    print(f"cửa sổ {calories[0:k]}: T={T:>3} -> {s:+d}, points={points}")
    for i in range(k, len(calories)):
        out, inn = calories[i - k], calories[i]
        T += inn - out
        s = score(T, lower, upper)
        points += s
        print(f"cửa sổ {calories[i - k + 1:i + 1]}: vào {inn}, ra {out}, T={T:>3} -> {s:+d}, points={points}")
    return points

trace_diet([1, 2, 3, 4, 5], 3, 6, 10)
print()
trace_diet([6, 5, 0, 0], 2, 1, 5)

cửa sổ [1, 2, 3]: T=  6 -> +0, points=0
cửa sổ [2, 3, 4]: vào 4, ra 1, T=  9 -> +0, points=0
cửa sổ [3, 4, 5]: vào 5, ra 2, T= 12 -> +1, points=1

cửa sổ [6, 5]: T= 11 -> +1, points=1
cửa sổ [5, 0]: vào 0, ra 6, T=  5 -> +0, points=1
cửa sổ [0, 0]: vào 0, ra 5, T=  0 -> -1, points=0


0

`f"{s:+d}"` in số nguyên **luôn kèm dấu** (`+1`, `-1`, `+0`) — tiện để đọc điểm.

## 7. Cách 3 — Prefix sum

Tính `pre` một lần; tổng cửa sổ bắt đầu ở `i` là `pre[i + k] - pre[i]`. Cũng `O(n)` nhưng tốn thêm `O(n)` bộ nhớ. Ưu điểm: trả lời tổng **đoạn bất kỳ** trong `O(1)`.

In [10]:
def diet_prefix(calories: List[int], k: int, lower: int, upper: int) -> int:
    pre = [0] + list(accumulate(calories))
    points = 0
    for i in range(len(calories) - k + 1):
        points += score(pre[i + k] - pre[i], lower, upper)
    return points

print(diet_prefix([1, 2, 3, 4, 5], 1, 3, 3), diet_prefix([3, 2], 2, 0, 1), diet_prefix([6, 5, 0, 0], 2, 1, 5))

0 1 0


## 8. Code nộp LeetCode

In [11]:
class Solution:
    def dietPlanPerformance(self, calories: List[int], k: int, lower: int, upper: int) -> int:
        T = sum(calories[:k])
        points = 0
        for i in range(k - 1, len(calories)):
            if i >= k:
                T += calories[i] - calories[i - k]
            if T < lower:
                points -= 1
            elif T > upper:
                points += 1
        return points

sol = Solution()
print(sol.dietPlanPerformance([1, 2, 3, 4, 5], 1, 3, 3))
print(sol.dietPlanPerformance([3, 2], 2, 0, 1))
print(sol.dietPlanPerformance([6, 5, 0, 0], 2, 1, 5))

0
1
0


Bản này gộp cửa sổ đầu vào cùng vòng lặp: `i` chạy từ `k - 1` (cuối cửa sổ đầu). Ở `i = k - 1` chưa trượt (vì `T` đã là tổng cửa sổ đầu); từ `i >= k` mới trượt. Nhờ vậy khối `if/elif` chỉ viết một lần.

In [12]:
class SolutionPrefix:
    def dietPlanPerformance(self, calories: List[int], k: int, lower: int, upper: int) -> int:
        pre = [0]
        for c in calories:
            pre.append(pre[-1] + c)
        points = 0
        for i in range(len(calories) - k + 1):
            T = pre[i + k] - pre[i]
            points += (T > upper) - (T < lower)    # True/False là 1/0
        return points

print(SolutionPrefix().dietPlanPerformance([6, 5, 0, 0], 2, 1, 5))
print((5 > 1) - (5 < 0), (0 > 5) - (0 < 1))   # mẹo bool → số

0
1 -1


Mẹo `(T > upper) - (T < lower)`: trong Python `True == 1`, `False == 0`, nên biểu thức cho `+1`, `-1` hoặc `0` mà không cần `if`.

## 9. Vì sao đúng? (invariant)

**Invariant:** ngay sau khi cập nhật ở bước có index cuối `i` (`k - 1 <= i <= n - 1`), `T = calories[i-k+1] + ... + calories[i]` — đúng tổng cửa sổ kết thúc tại `i`.

- **Khởi đầu** `i = k - 1`: `T = sum(calories[:k])` ✔.
- **Duy trì:** `T_mới = T_cũ + calories[i] - calories[i-k]` = tổng `[i-k+1 .. i]` (thêm phần tử mới bên phải, bỏ phần tử cũ nhất bên trái).
- **Phủ đủ:** `i` chạy hết `k - 1 .. n - 1` → đúng `n - k + 1` cửa sổ, mỗi cửa sổ chấm điểm **đúng một lần**.

Quy tắc chấm điểm áp dụng y hệt đề, nên tổng điểm đúng.

## 10. Độ phức tạp

Gọi `n = len(calories)`.

| Cách | Time | Space |
|---|---|---|
| Cộng lại từng cửa sổ | `O((n-k+1)·k)` → xấu nhất `O(n²)` | `O(k)` (slice) |
| Sliding window | `O(n)` | `O(1)` |
| Prefix sum | `O(n)` | `O(n)` |

Sliding window đọc mỗi phần tử tối đa 2 lần (một lần vào, một lần ra).

## 11. Bộ test: mẫu, biên, đối chiếu ngẫu nhiên

In [13]:
cases = [
    ([1, 2, 3, 4, 5], 1, 3, 3, 0),
    ([3, 2], 2, 0, 1, 1),
    ([6, 5, 0, 0], 2, 1, 5, 0),
    ([0], 1, 0, 0, 0),                 # đúng bằng lower = upper
    ([0], 1, 1, 1, -1),                # nhỏ hơn lower
    ([5], 1, 1, 4, 1),                 # lớn hơn upper
    ([1, 1, 1, 1], 4, 4, 4, 0),        # k = n: chỉ 1 cửa sổ
    ([0, 0, 0], 2, 1, 2, -2),
    ([20000] * 5, 3, 0, 59999, 3),
]
solvers = [diet_brute, diet_sliding, diet_prefix,
           Solution().dietPlanPerformance, SolutionPrefix().dietPlanPerformance]
for cal, k, lo, up, expected in cases:
    for f in solvers:
        assert f(cal, k, lo, up) == expected, (f, cal, k, lo, up)
    print(f"{str(cal):<26} k={k} [{lo},{up}] -> {expected:+d} PASS")

[1, 2, 3, 4, 5]            k=1 [3,3] -> +0 PASS
[3, 2]                     k=2 [0,1] -> +1 PASS
[6, 5, 0, 0]               k=2 [1,5] -> +0 PASS
[0]                        k=1 [0,0] -> +0 PASS
[0]                        k=1 [1,1] -> -1 PASS
[5]                        k=1 [1,4] -> +1 PASS
[1, 1, 1, 1]               k=4 [4,4] -> +0 PASS
[0, 0, 0]                  k=2 [1,2] -> -2 PASS
[20000, 20000, 20000, 20000, 20000] k=3 [0,59999] -> +3 PASS


In [14]:
import random
random.seed(1176)
for _ in range(1000):
    n = random.randint(1, 15)
    cal = [random.randint(0, 10) for _ in range(n)]
    k = random.randint(1, n)
    lo = random.randint(0, 30)
    up = random.randint(lo, 40)
    expected = diet_brute(cal, k, lo, up)
    for f in solvers[1:]:
        assert f(cal, k, lo, up) == expected
print("1000 test ngẫu nhiên khớp brute force. PASS")

big = [random.randint(0, 20000) for _ in range(10**5)]
t0 = time.perf_counter()
r = Solution().dietPlanPerformance(big, 50000, 10**8, 6 * 10**8)
print(f"n=10^5, k=5*10^4: kết quả {r}, mất {time.perf_counter() - t0:.3f} giây")

1000 test ngẫu nhiên khớp brute force. PASS
n=10^5, k=5*10^4: kết quả 0, mất 0.009 giây


## 12. Lỗi người mới hay gặp

### Lỗi 1: Sai số cửa sổ — `range(n - k)` thiếu cửa sổ cuối.

### Lỗi 2: Trừ nhầm phần tử ra: `calories[i - k + 1]` thay vì `calories[i - k]`.

### Lỗi 3: Quên chấm điểm cửa sổ **đầu tiên** (chỉ chấm trong vòng trượt).

### Lỗi 4: Dùng `<=`/`>=` thay vì `<`/`>` — `T == lower` hoặc `T == upper` phải **không** đổi điểm.

In [15]:
def wrong_range(cal, k, lo, up):
    return sum(score(sum(cal[i:i + k]), lo, up) for i in range(len(cal) - k))

def wrong_out_index(cal, k, lo, up):
    T = sum(cal[:k]); p = score(T, lo, up)
    for i in range(k, len(cal)):
        T += cal[i] - cal[i - k + 1]
        p += score(T, lo, up)
    return p

def wrong_skip_first(cal, k, lo, up):
    T = sum(cal[:k]); p = 0
    for i in range(k, len(cal)):
        T += cal[i] - cal[i - k]
        p += score(T, lo, up)
    return p

def wrong_inclusive(cal, k, lo, up):
    p = 0
    for i in range(len(cal) - k + 1):
        T = sum(cal[i:i + k])
        p += -1 if T <= lo else (1 if T >= up else 0)
    return p

args = ([6, 5, 0, 0], 2, 1, 5)
print("đúng:", Solution().dietPlanPerformance(*args))
print("thiếu cửa sổ cuối:", wrong_range(*args))
print("trừ nhầm index:", wrong_out_index(*args))
print("bỏ cửa sổ đầu:", wrong_skip_first(*args))
print("dùng <=, >=:", wrong_inclusive([1, 2, 3, 4, 5], 1, 3, 3), "(đúng là 0)")

đúng: 0
thiếu cửa sổ cuối: 1
trừ nhầm index: 3
bỏ cửa sổ đầu: -1
dùng <=, >=: -1 (đúng là 0)


`f(*args)`: dấu `*` **tách tuple** thành các đối số riêng — `f(*(a, b, c))` giống `f(a, b, c)`.

## 13. Mẹo nhận dạng pattern

- "Mọi đoạn con **liên tiếp** độ dài **cố định k**" → **sliding window cố định**: thêm phải, bớt trái.
- "Tổng của nhiều đoạn bất kỳ" → **prefix sum**.
- Brute force `sum(a[i:i+k])` trong vòng lặp là dấu hiệu `O(n·k)` cần tối ưu.

Bài cùng họ: 643 (Maximum Average Subarray I), 1343 (Number of Sub-arrays of Size K and Average ≥ Threshold), 1456 (Max Vowels in Substring of Given Length), 303 (Range Sum Query).

## 14. Biến thể

### 14.1. Tổng lớn nhất của cửa sổ k (643)

In [16]:
def max_window_sum(a: List[int], k: int) -> int:
    T = best = sum(a[:k])
    for i in range(k, len(a)):
        T += a[i] - a[i - k]
        best = max(best, T)
    return best

print(max_window_sum([1, 12, -5, -6, 50, 3], 4))   # 51 → trung bình 12.75

51


### 14.2. Đếm cửa sổ có trung bình ≥ ngưỡng (1343)
Trung bình ≥ `t` ⇔ tổng ≥ `t·k` — đổi về tổng số nguyên để tránh chia.

In [17]:
def count_windows_avg(a: List[int], k: int, threshold: int) -> int:
    target = threshold * k
    T = sum(a[:k])
    cnt = int(T >= target)
    for i in range(k, len(a)):
        T += a[i] - a[i - k]
        cnt += T >= target
    return cnt

print(count_windows_avg([2, 2, 2, 2, 5, 5, 5, 8], 3, 4))   # 3

3


### 14.3. So sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| Brute | cộng lại mỗi cửa sổ | `O(n·k)` | `O(k)` | TLE với n = 10^5 |
| Sliding window | +vào −ra | `O(n)` | `O(1)` | **nộp bài** |
| Prefix sum | `pre[i+k] - pre[i]` | `O(n)` | `O(n)` | tiện khi nhiều truy vấn đoạn |

## 15. Bài tập vận dụng (TODO)

### Bài 1 — Tính tay
`calories = [2, 7, 1, 8, 2, 8]`, `k = 2`, `lower = 5`, `upper = 9`. Liệt kê tổng các cửa sổ và tính điểm.

### Bài 2 — Liệt kê tổng mọi cửa sổ bằng sliding window

In [18]:
def window_sums_exercise(a: List[int], k: int) -> List[int]:
    # TODO: trả list n - k + 1 tổng, dùng T += a[i] - a[i - k]
    pass

# assert window_sums_exercise([1, 2, 3, 4, 5], 3) == [6, 9, 12]
# assert window_sums_exercise([7], 1) == [7]

### Bài 3 — Hoàn thiện lời giải bài chính

In [19]:
def diet_exercise(calories: List[int], k: int, lower: int, upper: int) -> int:
    # TODO: tổng cửa sổ đầu, chấm điểm; trượt và chấm điểm từng cửa sổ
    pass

# assert diet_exercise([1, 2, 3, 4, 5], 1, 3, 3) == 0
# assert diet_exercise([6, 5, 0, 0], 2, 1, 5) == 0

### Bài 4 — Prefix sum: tổng đoạn bất kỳ
Viết `build_prefix(a)` và `range_sum(pre, i, j)` trả tổng `a[i..j]` (bao gồm cả `j`).

In [20]:
def build_prefix_exercise(a: List[int]) -> List[int]:
    # TODO
    pass

def range_sum_exercise(pre: List[int], i: int, j: int) -> int:
    # TODO: chú ý j bao gồm → pre[j + 1] - pre[i]
    pass

# pre = build_prefix_exercise([-2, 0, 3, -5, 2, -1])
# assert range_sum_exercise(pre, 0, 2) == 1
# assert range_sum_exercise(pre, 2, 5) == -1

### Bài 5 — Số ngày "tốt"
Đếm số cửa sổ nằm **trong** khoảng `[lower, upper]` (không bị trừ, không được cộng).

In [21]:
def good_windows_exercise(calories: List[int], k: int, lower: int, upper: int) -> int:
    # TODO
    pass

# assert good_windows_exercise([1, 2, 3, 4, 5], 1, 3, 3) == 1
# assert good_windows_exercise([6, 5, 0, 0], 2, 1, 5) == 1

### Bài 6 — Mini challenge: điểm theo từng ngày bắt đầu
Trả về list điểm (`-1/0/+1`) của từng cửa sổ, rồi tìm **chuỗi dài nhất** các cửa sổ liên tiếp đều `0`.

In [22]:
def longest_zero_run_exercise(calories: List[int], k: int, lower: int, upper: int) -> int:
    # TODO: sinh list điểm bằng sliding window; đếm chuỗi 0 dài nhất
    pass

# assert longest_zero_run_exercise([1, 2, 3, 4, 5], 1, 2, 4) == 3
# assert longest_zero_run_exercise([6, 5, 0, 0], 2, 1, 5) == 1

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | có 5 cửa sổ | tổng: 9, 8, 9, 10, 10 | 0, 0, 0, +1, +1 |
| 2 | list kết quả | `T = sum(a[:k])` | `res.append(T)` sau mỗi lần trượt |
| 3 | xem mục 5 | hàm `score` | `points += score(T, lower, upper)` |
| 4 | `pre[0] = 0` | `pre.append(pre[-1] + x)` | `pre[j + 1] - pre[i]` |
| 5 | điều kiện ngược | `lower <= T <= upper` | đếm `+1` |
| 6 | dùng lại bài 2 | `cur`, `best` | gặp 0 thì `cur += 1`, khác thì `cur = 0` |

## 17. Đáp án tham khảo

**Bài 1:** cửa sổ `[2,7]=9` (0), `[7,1]=8` (0), `[1,8]=9` (0), `[8,2]=10` (+1), `[2,8]=10` (+1) → **2**.

In [23]:
def window_sums_answer(a: List[int], k: int) -> List[int]:
    T = sum(a[:k])
    res = [T]
    for i in range(k, len(a)):
        T += a[i] - a[i - k]
        res.append(T)
    return res

def diet_answer(calories: List[int], k: int, lower: int, upper: int) -> int:
    return sum(score(T, lower, upper) for T in window_sums_answer(calories, k))

def build_prefix_answer(a: List[int]) -> List[int]:
    pre = [0]
    for x in a:
        pre.append(pre[-1] + x)
    return pre

def range_sum_answer(pre: List[int], i: int, j: int) -> int:
    return pre[j + 1] - pre[i]

def good_windows_answer(calories: List[int], k: int, lower: int, upper: int) -> int:
    return sum(1 for T in window_sums_answer(calories, k) if lower <= T <= upper)

def longest_zero_run_answer(calories: List[int], k: int, lower: int, upper: int) -> int:
    best = cur = 0
    for T in window_sums_answer(calories, k):
        if score(T, lower, upper) == 0:
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    return best

assert Solution().dietPlanPerformance([2, 7, 1, 8, 2, 8], 2, 5, 9) == 2
assert window_sums_answer([1, 2, 3, 4, 5], 3) == [6, 9, 12]
assert window_sums_answer([7], 1) == [7]
assert diet_answer([1, 2, 3, 4, 5], 1, 3, 3) == 0
assert diet_answer([6, 5, 0, 0], 2, 1, 5) == 0
pre = build_prefix_answer([-2, 0, 3, -5, 2, -1])
assert range_sum_answer(pre, 0, 2) == 1
assert range_sum_answer(pre, 2, 5) == -1
assert good_windows_answer([1, 2, 3, 4, 5], 1, 3, 3) == 1
assert good_windows_answer([6, 5, 0, 0], 2, 1, 5) == 1
assert longest_zero_run_answer([1, 2, 3, 4, 5], 1, 2, 4) == 3
assert longest_zero_run_answer([6, 5, 0, 0], 2, 1, 5) == 1
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def dietPlanPerformance(self, calories, k, lower, upper):
        T = sum(calories[:k])
        points = 0
        for i in range(k - 1, len(calories)):
            if i >= k:
                T += calories[i] - calories[i - k]
            if T < lower:   points -= 1
            elif T > upper: points += 1
        return points
```

**Template sliding window cố định:**
```python
T = sum(a[:k]); xử_lý(T)
for i in range(k, n):
    T += a[i] - a[i - k]; xử_lý(T)
```

**Checklist:**
- [ ] Đủ `n - k + 1` cửa sổ (kể cả cửa sổ đầu và cuối)?
- [ ] Phần tử ra là `a[i - k]` khi phần tử vào là `a[i]`?
- [ ] So sánh **nghiêm ngặt** `<` và `>`?
- [ ] Test `k = 1`, `k = n`, tổng đúng bằng `lower`/`upper`?